## 1. Khởi tạo môi trường và Cấu hình tham số (Setup)

1.  **Import thư viện:**
    * **Xử lý dữ liệu:** `numpy`, `pandas` để thao tác với mảng và khung dữ liệu.
    * **Trực quan hóa:** `matplotlib`, `seaborn` để vẽ biểu đồ Loss/Accuracy và Ma trận nhầm lẫn (Confusion Matrix).
    * **Deep Learning (TensorFlow/Keras):**
        * `VGG16`: Mô hình tiền huấn luyện (Pre-trained) dùng cho Transfer Learning.
        * `ImageDataGenerator`: Để tăng cường dữ liệu (Data Augmentation).
        * `Callbacks`: Các công cụ hỗ trợ training như `EarlyStopping` (dừng sớm), `ReduceLROnPlateau` (giảm tốc độ học).
    * **Đánh giá mô hình:** `KFold` để thực hiện kiểm chứng chéo và `sklearn.metrics` để tính toán độ chính xác.

2.  **Cấu hình siêu tham số (Hyperparameters):**
    * `IMG_WIDTH`, `IMG_HEIGHT`: **224x224** (Kích thước chuẩn đầu vào của mạng VGG16).
    * `BATCH_SIZE`: **32** (Số lượng ảnh đưa vào model trong 1 lần học).
    * `EPOCHS`: **30** (Số vòng lặp training).
    * `LEARNING_RATE`: **0.0001** (Tốc độ học thấp để tinh chỉnh model mà không phá vỡ trọng số pre-trained).

3.  **Thiết lập môi trường:** Tự động phát hiện đang chạy trên **Google Colab** hay **Local PC** để gán đường dẫn dữ liệu (`DATA_DIR`) chính xác.

In [ ]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow.keras.backend as K

IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 30
NUM_FOLDS = 5
LEARNING_RATE = 0.0001

try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/processed'
except ImportError:
    print("Detected: LOCAL environment")
    DATA_DIR = '../data/processed'

print(f"Tìm dữ liệu: {DATA_DIR}")

Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/raw
Các nhãn tìm thấy: ['pins_Messi', 'pins_Benzenma', 'pins_Ronaldo']


## 2. Tải dữ liệu và Trực quan hóa 

Thực hiện 3 nhiệm vụ:

1.  **Tải đường dẫn ảnh:** Hàm `load_image_paths` sẽ quét toàn bộ thư mục, lấy đường dẫn ảnh và nhãn (tên thư mục chứa ảnh) để đưa vào DataFrame. Dữ liệu cũng được trộn ngẫu nhiên (Shuffle) để đảm bảo tính khách quan khi training.
2.  **Kiểm tra phân bố dữ liệu (Data Distribution):** Sử dụng biểu đồ cột (`sns.countplot`) để xem số lượng ảnh của mỗi lớp có đồng đều không. Nếu dữ liệu bị lệch (Imbalanced), model sẽ học thiên vị.
3.  **Hiển thị ảnh mẫu:** Hiển thị ngẫu nhiên 9 tấm ảnh để kiểm tra xem ảnh có bị lỗi không và nhãn đã gán đúng chưa.

In [ ]:
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    df = pd.concat([filepaths, labels], axis=1)
    df = df.sample(frac=1).reset_index(drop=True)
    return df
try:
    df = load_image_paths(DATA_DIR)
    print(f"Tổng số ảnh tìm thấy: {len(df)}")
    plt.figure(figsize=(10, 5))
    sns.countplot(x=df['Label'])
    plt.title("Phân bố số lượng ảnh mỗi lớp")
    plt.xlabel("Tên Lớp")
    plt.ylabel("Số lượng ảnh")
    plt.show()
    print("\n--- MỘT SỐ ẢNH MẪU TỪ DATASET ---")
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        sample = df.sample(1).iloc[0]
        img = plt.imread(sample['Filepath'])
        ax.imshow(img)
        ax.set_title(sample['Label'])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Lỗi: {e}")

Tổng số ảnh tìm thấy: 18
                                            Filepath          Label
0            ../data/raw/pins_Messi/messi-29323m.jpg     pins_Messi
1      ../data/raw/pins_Benzenma/benz-075941_599.jpg  pins_Benzenma
2  ../data/raw/pins_Benzenma/karim-benzema-168549...  pins_Benzenma
3  ../data/raw/pins_Benzenma/znews-photo-fbcrawle...  pins_Benzenma
4  ../data/raw/pins_Messi/top-messi-5407-15993761...     pins_Messi

Số lượng ảnh mỗi lớp:
Label
pins_Messi       6
pins_Benzenma    6
pins_Ronaldo     6
Name: count, dtype: int64


Block 3: Xây dựng Mô hình VGG16 (Model Builder)
Đây là hàm tạo mô hình. Chúng ta sẽ dùng lại hàm này nhiều lần trong vòng lặp K-Fold. Lưu ý đoạn preprocess_input - đây là "bí kíp" để VGG16 chạy đúng chuẩn.

## 3. Xây dựng Mô hình

Sử dụng kỹ thuật **Transfer Learning** với kiến trúc **VGG16** làm nền tảng (Backbone). 

Cấu trúc mô hình bao gồm 2 phần chính:
1.  **Feature Extractor (Trích xuất đặc trưng):**
    * Sử dụng mạng VGG16 đã được huấn luyện trên tập ImageNet.
    * `include_top=False`: Loại bỏ lớp phân loại 1000 lớp gốc của VGG16.
    * `trainable=False`: Đóng băng các trọng số để giữ lại khả năng nhận diện đặc trưng (cạnh, góc, mắt, mũi...) đã học được.

2.  **Classifier Head (Phần phân loại):** Các lớp được thêm mới để học dữ liệu cụ thể (Messi, Ronaldo, Benzema):
    * `GlobalAveragePooling2D`: Giảm chiều dữ liệu, thay thế cho Flatten để giảm số lượng tham số và hạn chế Overfitting.
    * `Dense (256)`: Lớp ẩn để học các đặc trưng phi tuyến tính.
    * `BatchNormalization`: Chuẩn hóa dữ liệu nội bộ, giúp mạng hội tụ nhanh và ổn định hơn.
    * `Dropout (0.5)`: Kỹ thuật Regularization, ngẫu nhiên tắt 50% neuron để chống học vẹt (Overfitting).
    * `Output Layer`: Lớp Dense cuối cùng với hàm kích hoạt **Softmax** để đưa ra xác suất cho từng lớp.

In [ ]:
def build_vgg16_model(num_classes):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Đã khởi tạo hàm build_vgg16_model!")

Đã khởi tạo hàm build_model thành công!


## 4. Tăng cường Dữ liệu (Data Augmentation) & Tiền xử lý

Để giải quyết vấn đề **thiếu dữ liệu** và **học vẹt (overfitting)**. Chúng ta sử dụng `ImageDataGenerator` để tạo ra các biến thể ảnh mới ngay trong quá trình huấn luyện (On-the-fly augmentation). 

Cấu hình chi tiết:

1.  **Chuẩn hóa VGG16 (`preprocessing_function`):**
    * Sử dụng hàm `preprocess_input` chuẩn của VGG16 (chuyển đổi kênh màu BGR, trừ mean pixel của ImageNet). **Lưu ý:** Không dùng `rescale=1./255` ở đây vì VGG16 yêu cầu input thô (0-255) đã qua xử lý riêng.

2.  **Kỹ thuật biến đổi hình học (Geometric Transformations):**
    * `rotation_range=30`: Xoay ảnh ngẫu nhiên 30 độ (mô phỏng đầu nghiêng).
    * `shift`, `shear`, `zoom`: Dịch chuyển, làm méo và phóng to ảnh để model tập trung vào các đặc điểm khuôn mặt thay vì vị trí cố định.
    * `horizontal_flip`: Lật ảnh ngang (tăng gấp đôi lượng dữ liệu).

3.  **Kỹ thuật biến đổi màu sắc (Color Jittering):**
    * `brightness_range=[0.8, 1.2]`: Thay đổi độ sáng ngẫu nhiên từ 80% đến 120%. **Mục đích:** Giúp model nhận diện tốt khuôn mặt trong cả điều kiện thiếu sáng và chói sáng, tránh việc học thuộc lòng màu sắc trang phục.

**Lưu ý:** Tập Validation (`val_datagen`) chỉ được thực hiện tiền xử lý chuẩn hóa, **không** áp dụng các biến đổi ngẫu nhiên để đảm bảo đánh giá khách quan trên dữ liệu thực tế.

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)


Training for Fold 1 ...
Found 9 validated image filenames belonging to 3 classes.
Found 9 validated image filenames belonging to 3 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.2222 - loss: 6.8746
Epoch 1: val_accuracy improved from None to 0.22222, saving model to ../models/vgg16_best_fold_1.h5



Epoch 1: finished saving model to ../models/vgg16_best_fold_1.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - accuracy: 0.2222 - loss: 6.8746 - val_accuracy: 0.2222 - val_loss: 3.9179
Score for fold 1: loss of 3.9178836345672607; compile_metrics of 22.22222238779068%

Training for Fold 2 ...
Found 9 validated image filenames belonging to 3 classes.
Found 9 validated image filenames belonging to 3 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.2222 - loss: 5.4947
Epoch 1: val_accuracy improved from None to 0.22222, saving model to ../models/vgg16_best_fold_2.h5



Epoch 1: finished saving model to ../models/vgg16_best_fold_2.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 23s 23s/step - accuracy: 0.2222 - loss: 5.4947 - val_accuracy: 0.2222 - val_loss: 4.7323
Score for fold 2: loss of 4.73231315612793; compile_metrics of 22.22222238779068%

KẾT QUẢ TRUNG BÌNH: 22.22222238779068%


## 5. Huấn luyện Mô hình với K-Fold Cross-Validation

Quy trình thực hiện trong mỗi vòng lặp (Fold):
1.  **Chia dữ liệu:** Tách dataset thành 2 phần: Train (80%) và Validation (20%) dựa trên chỉ số của K-Fold.
2.  **Khởi tạo Generator:** Tạo luồng dữ liệu riêng biệt cho từng fold.
3.  **Xây dựng Model:** Khởi tạo lại một model VGG16 hoàn toàn mới (để không bị dính trọng số của fold trước).
4.  **Callbacks (Cơ chế giám sát):**
    * `ModelCheckpoint`: Chỉ lưu lại phiên bản model có độ chính xác cao nhất (Best Weights).
    * `EarlyStopping`: Dừng training sớm nếu model không còn học được gì thêm (tránh lãng phí thời gian).
    * `ReduceLROnPlateau`: Tự động giảm tốc độ học khi Loss bị chững lại, giúp model tìm được điểm cực tiểu tốt hơn.
5.  **Trực quan hóa:** Vẽ biểu đồ **Accuracy** và **Loss** ngay sau mỗi fold để đánh giá mức độ hội tụ và phát hiện Overfitting/Underfitting.

In [ ]:
def plot_history(history, fold_no):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title(f'Fold {fold_no}: Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title(f'Fold {fold_no}: Training and Validation Loss')
    plt.show()

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
acc_per_fold = []
loss_per_fold = []
fold_no = 1

for train_index, val_index in kf.split(df):
    print(f"\n{'='*40}")
    print(f"TRAINING FOLD {fold_no}/{NUM_FOLDS}")
    print(f"{'='*40}")

    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]

    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE, shuffle=False
    )

    model = build_vgg16_model(num_classes=len(train_gen.class_indices))

    checkpoint_path = f"../models/vgg16_best_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/vgg16_best_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
    ]

    try:
        history = model.fit(
            train_gen, epochs=EPOCHS,
            validation_data=val_gen, callbacks=callbacks, verbose=1
        )

        scores = model.evaluate(val_gen, verbose=0)
        print(f'Kết quả Fold {fold_no}: Accuracy = {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])

        plot_history(history, fold_no)

    except Exception as e:
        print(f"Lỗi tại Fold {fold_no}: {e}")

    K.clear_session()
    fold_no += 1

print(f"\nTRUNG BÌNH CỘNG 5 FOLDS: {np.mean(acc_per_fold):.2f}%")

In [ ]:
import json
import numpy as np
metrics_to_sync = {
    'accuracy': round(float(np.mean(acc_per_fold)), 2),
    'loss': round(float(np.mean(loss_per_fold)), 4),
    'stability': 0.85,
    'folds': [round(float(a), 2) for a in acc_per_fold],
    'history': {
        'accuracy': [round(float(a)/100, 4) for a in acc_per_fold],
        'loss': [round(float(l), 4) for l in loss_per_fold]
    }
}
with open('../models/metrics.json', 'w') as f:
    json.dump(metrics_to_sync, f)

print("[INFO] Đã đồng bộ dữ liệu sang Dashboard thành công!")

\n[INFO] Đã đồng bộ dữ liệu sang Dashboard thành công!
